# PathSnap Step 3: HMM and Viterbi Route Reconstruction (Himesh)

Decodes a route for each trip with a Hidden Markov Model over candidate arcs:

- **Hidden state:** the road arc the vehicle is on at each GPS observation (the candidate set from Juhi's features).
- **Emission:** Gaussian on point-to-arc distance, `-d² / (2σ²)` in log space.
- **Transition:** stay on the same arc (`p_stay`), move to a connected arc (`p_conn`, head node of one arc equals tail node of the next), or any other move (`p_other`).
- **Viterbi** finds the most likely arc sequence per segment.
- **Gaps:** a segment breaks when there is no candidate or when the time gap is over 30 s. The route is not forced across gaps.

It compares the **nearest-candidate baseline** with **HMM/Viterbi** on pointwise accuracy and route continuity.

**Important:** the emission uses distance only for now. When Gagana's classifier is ready, replace the emission with its probabilities (`-log p`).

**Attach data first:** Add data → *PathSnap Student Data*. Then Run All. Expected run time: 3 to 6 minutes.

In [10]:
import os, glob, json, time
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

def find_root():
    env = os.environ.get("PATHSNAP_ROOT")
    if env and os.path.isdir(os.path.join(env, "train")):
        return env
    def search():
        hits = glob.glob("/kaggle/input/**/train", recursive=True) + glob.glob("/kaggle/working/raw/**/train", recursive=True)
        for h in hits:
            parent = os.path.dirname(h)
            if os.path.isdir(h) and os.path.isdir(os.path.join(parent, "student_test")):
                return parent
        return None
    root = search()
    if root is None:
        zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        if zips:
            import zipfile
            zipfile.ZipFile(zips[0]).extractall("/kaggle/working/raw")
            root = search()
    if root is None:
        raise FileNotFoundError("Attach the PathSnap data: a folder with train/ and student_test/, or Student_folder.zip")
    return root

ROOT = find_root()
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "outputs"
os.makedirs(OUT, exist_ok=True)
print("ROOT:", ROOT)
print("OUT :", OUT)

ROOT: /kaggle/input/datasets/juhishahh/pathsnap-student-data/Student folder
OUT : /kaggle/working


In [11]:
R_CAND = 50.0    # metres: candidate radius and label radius
K_CAND = 40      # nearest arc midpoints examined per GPS point
FEATS = ["dist", "hd", "alen", "speed", "dt"]

def load_trip(folder, tid, with_route=False):
    d = os.path.join(folder, tid)
    track = np.loadtxt(os.path.join(d, tid + ".track"), ndmin=2)
    nodes = np.loadtxt(os.path.join(d, tid + ".nodes"), ndmin=2)
    arcs = np.loadtxt(os.path.join(d, tid + ".arcs"), dtype=int, ndmin=2)
    route = None
    if with_route:
        route = np.loadtxt(os.path.join(d, tid + ".route"), dtype=int, ndmin=1)
    return track, nodes, arcs, route

def project(lon, lat, lat0):
    # local equirectangular projection in metres (city scale)
    return lon * np.cos(np.radians(lat0)) * 111320.0, lat * 110574.0

def seg_dist(px, py, ax, ay, bx, by):
    abx, aby = bx - ax, by - ay
    t = np.clip(((px - ax) * abx + (py - ay) * aby) / (abx ** 2 + aby ** 2 + 1e-9), 0, 1)
    return np.hypot(px - (ax + t * abx), py - (ay + t * aby))

def connectivity(nodes, arcs):
    n = len(nodes)
    valid = (arcs[:, 0] >= 0) & (arcs[:, 0] < n) & (arcs[:, 1] >= 0) & (arcs[:, 1] < n)
    a = arcs[valid]
    g = coo_matrix((np.ones(len(a)), (a[:, 0], a[:, 1])), shape=(n, n))
    ncomp, _ = connected_components(g, directed=True, connection="weak")
    return ncomp

def candidates(track, nodes, arcs):
    """Candidate arcs per GPS observation. Does NOT use the reference route."""
    lat0 = track[:, 1].mean()
    nx, ny = project(nodes[:, 0], nodes[:, 1], lat0)
    s, e = arcs[:, 0], arcs[:, 1]
    ax, ay, bx, by = nx[s], ny[s], nx[e], ny[e]
    mx, my = (ax + bx) / 2, (ay + by) / 2
    alen = np.hypot(bx - ax, by - ay)
    abear = np.arctan2(bx - ax, by - ay)
    px, py = project(track[:, 0], track[:, 1], lat0)
    n = len(px)
    dx = np.r_[np.diff(px), 0.0]
    dy = np.r_[np.diff(py), 0.0]
    dt = np.r_[np.diff(track[:, 2]), 1.0]
    dt = np.where(dt <= 0, 1.0, dt)
    speed = np.hypot(dx, dy) / dt
    heading = np.arctan2(dx, dy)
    k = min(K_CAND, len(arcs))
    _, idx = cKDTree(np.column_stack([mx, my])).query(np.column_stack([px, py]), k=k)
    idx = np.asarray(idx).reshape(n, k)
    obs = np.repeat(np.arange(n), k)
    arc = idx.ravel()
    d = seg_dist(px[obs], py[obs], ax[arc], ay[arc], bx[arc], by[arc])
    hd = np.abs(np.angle(np.exp(1j * (heading[obs] - abear[arc]))))
    keep = d <= R_CAND
    return pd.DataFrame({
        "obs": obs[keep], "arc": arc[keep], "dist": d[keep], "hd": hd[keep],
        "alen": alen[arc[keep]], "speed": speed[obs[keep]], "dt": dt[obs[keep]],
    })

def reference_labels(track, nodes, arcs, route):
    """TRAINING AND SCORING ONLY. Nearest reference-route arc per GPS point (proxy label)."""
    lat0 = track[:, 1].mean()
    nx, ny = project(nodes[:, 0], nodes[:, 1], lat0)
    s, e = arcs[:, 0], arcs[:, 1]
    px, py = project(track[:, 0], track[:, 1], lat0)
    best = np.full(len(px), np.inf)
    lab = np.full(len(px), -1)
    for a in np.unique(route):
        d = seg_dist(px, py, nx[s[a]], ny[s[a]], nx[e[a]], ny[e[a]])
        m = d < best
        best[m] = d[m]
        lab[m] = a
    lab[best > R_CAND] = -1
    return lab

def trip_frames(folder, tid, with_route=False):
    track, nodes, arcs, route = load_trip(folder, tid, with_route)
    cand = candidates(track, nodes, arcs)
    cand.insert(0, "trip", tid)
    obs = pd.DataFrame({"trip": tid, "obs": np.arange(len(track)),
                        "lon": track[:, 0], "lat": track[:, 1], "t": track[:, 2]})
    if with_route:
        obs["label"] = reference_labels(track, nodes, arcs, route)
        cand = cand.merge(obs[["obs", "label"]], on="obs")
        cand["y"] = ((cand["arc"] == cand["label"]) & (cand["label"] >= 0)).astype(int)
    return cand, obs

In [12]:
def trips_in(part):
    base = os.path.join(ROOT, part)
    return sorted(d for d in os.listdir(base) if os.path.isdir(os.path.join(base, d)))

def build_feature_table(part):
    cs, os_ = [], []
    for tid in trips_in(part):
        c, o = trip_frames(os.path.join(ROOT, part), tid, with_route=True)
        cs.append(c)
        os_.append(o)
    return pd.concat(cs, ignore_index=True), pd.concat(os_, ignore_index=True)

def prepare(part):
    items = []
    folder = os.path.join(ROOT, part)
    for tid in trips_in(part):
        track, nodes, arcs, _ = load_trip(folder, tid)
        cand, obs = trip_frames(folder, tid, with_route=True)
        groups = {k: (v["arc"].values, v["dist"].values) for k, v in cand.groupby("obs")}
        base = np.full(len(obs), -1)
        idx = cand.groupby("obs")["dist"].idxmin()
        base[cand.loc[idx, "obs"].values] = cand.loc[idx, "arc"].values
        items.append(dict(tid=tid, arcs=arcs, groups=groups, t=obs["t"].values,
                          N=len(obs), lab=obs["label"].values, base=base))
    return items

train_items = prepare("train")
test_items = prepare("student_test")
print("prepared trips:", len(train_items), "train |", len(test_items), "student_test")

prepared trips: 80 train | 10 student_test


In [13]:
def decode_trip(groups, arcs, t, N, sigma, p_stay, p_conn, p_other, gap_s):
    head, tail = arcs[:, 1], arcs[:, 0]
    pred = np.full(N, -1)
    if not groups:
        return pred
    ids = np.array(sorted(groups.keys()))

    def run(seg):
        A = [groups[i][0] for i in seg]
        D = [groups[i][1] for i in seg]
        em = [-(d ** 2) / (2 * sigma ** 2) for d in D]
        V = [em[0]]
        B = []
        for k in range(1, len(seg)):
            prev, cur = A[k - 1], A[k]
            same = prev[:, None] == cur[None, :]
            conn = head[prev][:, None] == tail[cur][None, :]
            T = np.where(same, np.log(p_stay), np.where(conn, np.log(p_conn), np.log(p_other)))
            S = V[-1][:, None] + T
            B.append(S.argmax(axis=0))
            V.append(S.max(axis=0) + em[k])
        st = [int(V[-1].argmax())]
        for k in range(len(seg) - 1, 0, -1):
            st.append(int(B[k - 1][st[-1]]))
        st = st[::-1]
        for k, i in enumerate(seg):
            pred[i] = A[k][st[k]]

    seg = [ids[0]]
    for a, b in zip(ids[:-1], ids[1:]):
        if b == a + 1 and t[b] - t[a] <= gap_s:
            seg.append(b)
        else:
            run(seg)
            seg = [b]
    run(seg)
    return pred

def continuity(pred, arcs, t, gap_s):
    a, b = pred[:-1], pred[1:]
    m = (a >= 0) & (b >= 0) & (np.diff(t) <= gap_s)
    a, b = a[m], b[m]
    ok = (a == b) | (arcs[a, 1] == arcs[b, 0])
    return int(ok.sum()), int(len(a))

def run_items(items, sigma, p_stay=0.5, p_conn=0.4, p_other=1e-4, gap_s=30):
    n = hb = hh = okb = totb = okh = toth = 0
    for it in items:
        hm = decode_trip(it["groups"], it["arcs"], it["t"], it["N"], sigma, p_stay, p_conn, p_other, gap_s)
        lab = it["lab"]
        L = lab >= 0
        n += int(L.sum())
        hb += int((it["base"][L] == lab[L]).sum())
        hh += int((hm[L] == lab[L]).sum())
        a, b = continuity(it["base"], it["arcs"], it["t"], gap_s); okb += a; totb += b
        a, b = continuity(hm, it["arcs"], it["t"], gap_s); okh += a; toth += b
    return {"labelled_obs": n, "pointwise_baseline": hb / max(n, 1), "pointwise_hmm": hh / max(n, 1),
            "continuity_baseline": okb / max(totb, 1), "continuity_hmm": okh / max(toth, 1)}

In [14]:
# Decoder parameters are chosen on the TRAINING trips only.
grid = []
for sigma in [5.0, 10.0, 20.0]:
    for p_other in [1e-3, 1e-5]:
        r = run_items(train_items, sigma, p_other=p_other)
        grid.append({"sigma_m": sigma, "p_other": p_other, **r})
grid = pd.DataFrame(grid)
grid.to_csv(os.path.join(OUT, "hmm_grid_train.csv"), index=False)
print(grid.to_string(index=False))
best = grid.sort_values("pointwise_hmm", ascending=False).iloc[0]
print("chosen on train:", best.to_dict())

 sigma_m  p_other  labelled_obs  pointwise_baseline  pointwise_hmm  continuity_baseline  continuity_hmm
     5.0  0.00100        185863             0.55818       0.850621              0.82444        0.990184
     5.0  0.00001        185863             0.55818       0.855770              0.82444        0.993135
    10.0  0.00100        185863             0.55818       0.868236              0.82444        0.995801
    10.0  0.00001        185863             0.55818       0.869237              0.82444        0.996689
    20.0  0.00100        185863             0.55818       0.876038              0.82444        0.997488
    20.0  0.00001        185863             0.55818       0.877555              0.82444        0.997763
chosen on train: {'sigma_m': 20.0, 'p_other': 1e-05, 'labelled_obs': 185863.0, 'pointwise_baseline': 0.5581799497479326, 'pointwise_hmm': 0.8775549732867758, 'continuity_baseline': 0.8244403192282672, 'continuity_hmm': 0.9977628635346756}


In [15]:
test_res = run_items(test_items, best.sigma_m, p_other=best.p_other)
summary = pd.DataFrame([
    {"partition": "train (chosen params)", **run_items(train_items, best.sigma_m, p_other=best.p_other)},
    {"partition": "student_test", **test_res},
])
summary.to_csv(os.path.join(OUT, "hmm_vs_baseline.csv"), index=False)
print(summary.to_string(index=False))

            partition  labelled_obs  pointwise_baseline  pointwise_hmm  continuity_baseline  continuity_hmm
train (chosen params)        185863            0.558180       0.877555             0.824440        0.997763
         student_test         19133            0.552292       0.874301             0.825405        0.998636


## Notes for the report

- Hidden states, emission and transition formulas, and the Viterbi recurrence are the design in the first cell. Explain them in your own words.
- Missing candidates and gaps over 30 s break the segment, so no route is forced across a gap.
- Report pointwise accuracy and continuity for baseline vs HMM. Say when HMM did not help.
- Labels are the proxy labels from Juhi's notebook.